# LLM Pre-Filter for Company Retrieval
## Comparing Llama-3.1-8B vs Mistral-3B (local via Ollama)

**What this notebook does:**

Instead of searching all ~99k companies for every query, we first use a small LLM
to extract structured filters from the query (country, industry, NAICS code, keywords).
We then reduce the corpus to only matching companies before running BGE retrieval.

**Both LLMs run locally via [Ollama](https://ollama.com) — no external API needed.**

Before running this notebook, make sure:
```bash
# 1. Install Ollama (if not already installed)
#    https://ollama.com/download

# 2. Start Ollama
ollama serve

# 3. Pull both models (one-time download)
ollama pull llama3.1:8b      # ~4.7GB
ollama pull mistral:3b       # ~2.0GB
```

**Two LLMs compared:**
| Model | Size | Download | Speed | Expected quality |
|---|---|---|---|---|
| `llama3.1:8b` | 8B params | ~4.7GB | Slower | Better instruction following |
| `mistral:3b` | 3B params | ~2.0GB | Faster | Lightweight, efficient |

**Expected benefits of pre-filtering:**
- Higher Precision@k — fewer irrelevant companies in search space
- Lower search latency — BGE searches 3k companies instead of 99k
- Solves vocabulary mismatch (e.g. investment banks showing for 'software companies')

**Folder structure:**
```
result/
└── prefilter/
    ├── extracted_filters_llama.json       # LLM-extracted filters (Llama)
    ├── extracted_filters_ministral.json   # LLM-extracted filters (Ministral)
    ├── retrieval_llama.csv                # BGE retrieval on Llama-filtered corpus
    ├── retrieval_ministral.csv            # BGE retrieval on Ministral-filtered corpus
    ├── timing_llama.csv                   # Per-query latency breakdown (Llama)
    ├── timing_ministral.csv               # Per-query latency breakdown (Ministral)
    ├── evaluation_llama.csv               # Full metrics (Llama pipeline)
    ├── evaluation_ministral.csv           # Full metrics (Ministral pipeline)
    ├── comparison_prefilter.csv           # Side-by-side vs BGE baseline
    └── winner_llm.txt                     # Which LLM won — used by future notebooks
```

## 1 · Environment Setup & Folder Structure

In [1]:
import os, json, time, re
import numpy as np
import pandas as pd
import faiss
import torch
import requests
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

load_dotenv()

# ── Local LLM via Ollama ──────────────────────────────────────────────────────
# Both Llama and Ministral run locally via Ollama.
# Make sure Ollama is running: `ollama serve`
# Pull the models first:
#   ollama pull llama3.1:8b
#   ollama pull mistral:3b
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')

LLAMA_MODEL     = 'llama3.1:8b'   # exact name as shown by `ollama list`
MINISTRAL_MODEL = 'mistral:3b'    # exact name as shown by `ollama list`

def ollama_chat(prompt, model, system=None, temperature=0.0, max_tokens=150):
    """
    Call a local Ollama model.
    Uses the /api/chat endpoint which is compatible with all Ollama models.
    """
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})

    response = requests.post(
        f'{OLLAMA_BASE_URL}/api/chat',
        json={
            'model':   model,
            'messages': messages,
            'stream':  False,
            'options': {
                'temperature': temperature,
                'num_predict': max_tokens,
            }
        },
        timeout=60
    )
    response.raise_for_status()
    return response.json()['message']['content'].strip()

# ── Quick connectivity check ──────────────────────────────────────────────────
try:
    models_resp = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5)
    available   = [m['name'] for m in models_resp.json().get('models', [])]
    print(f'Ollama is running at {OLLAMA_BASE_URL}')
    print(f'Available models: {available}')
    if LLAMA_MODEL not in available:
        print(f'WARNING: {LLAMA_MODEL} not found — run: ollama pull {LLAMA_MODEL}')
    if MINISTRAL_MODEL not in available:
        print(f'WARNING: {MINISTRAL_MODEL} not found — run: ollama pull {MINISTRAL_MODEL}')
except Exception as e:
    print(f'ERROR: Cannot connect to Ollama at {OLLAMA_BASE_URL}')
    print(f'  Make sure Ollama is running: ollama serve')
    print(f'  Error: {e}')

# ── GPU check ─────────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# ── Output folder structure ───────────────────────────────────────────────────
RESULT_DIR = Path('result/4_llm_prefilter')
for folder in [
    RESULT_DIR,
    RESULT_DIR / 'filtered_corpus_llama',
    RESULT_DIR / 'filtered_corpus_ministral',
]:
    folder.mkdir(parents=True, exist_ok=True)

print(f'\nResult folders created under {RESULT_DIR}/')

Ollama is running at http://localhost:11434
Available models: ['llama3.1:8b']

Device : cpu

Result folders created under result/4_llm_prefilter/


## 2 · Load Corpus, Queries & BGE Model

We reuse:
- `result/company_corpus.csv` — de-duplicated company list from baseline experiments
- `result/embeddings_bge.npy` — pre-computed BGE embeddings (saves re-encoding time)
- `dataset/goi_search_results.json` — the 101 queries
- `dataset/production_results.xlsx` — for pseudo-relevance evaluation labels

BGE is used as the retrieval model because it was the best baseline.

In [ ]:
# ── Load corpus ──────────────────────────────────────────────────────────────
all_companies = pd.read_csv('result/company_corpus.csv')
summaries     = all_companies['summary'].fillna('').tolist()
print(f'Total companies : {len(all_companies):,}')
print(f'Columns available: {list(all_companies.columns)}')

# ── Load queries ──────────────────────────────────────────────────────────────
with open('dataset/goi_search_results.json', 'r') as f:
    data = json.load(f)
print(f'Total queries   : {len(data)}')

# ── Load production results for evaluation ────────────────────────────────────
production_df = pd.read_excel('dataset/production_results.xlsx')

# ── Load BGE model & pre-computed embeddings ─────────────────────────────────
print('\nLoading BGE model...')
model_bge = SentenceTransformer('BAAI/bge-large-en-v1.5', device=DEVICE)

print('Loading pre-computed BGE embeddings...')
all_embeddings = np.load('result/3_baseline_BGE/company_embeddings.npy').astype('float32')
print(f'Embeddings shape: {all_embeddings.shape}')  # (98716, 1024)

## 3 · Inspect Available Filter Fields

Before building the LLM filter, we need to understand what structured fields
are available in the corpus that we can actually filter on.

Check which columns have good coverage (low % missing) and are useful for filtering.

In [ ]:
# ── Check field coverage ─────────────────────────────────────────────────────
print('=== CORPUS FIELD COVERAGE ===')
for col in all_companies.columns:
    n_missing  = all_companies[col].isna().sum()
    n_empty    = (all_companies[col].astype(str).str.strip() == '').sum()
    pct_filled = 100 * (1 - (n_missing + n_empty) / len(all_companies))
    n_unique   = all_companies[col].nunique()
    print(f'  {col:<30} filled={pct_filled:5.1f}%  unique={n_unique:,}')

# ── Sample values for filterable fields ──────────────────────────────────────
print('\n=== SAMPLE VALUES ===')
filter_candidates = ['country', 'industry', 'naics_code', 'naics_description',
                      'region', 'city', 'tags', 'type', 'sector']
for col in filter_candidates:
    if col in all_companies.columns:
        samples = all_companies[col].dropna().unique()[:8]
        print(f'  {col}: {list(samples)}')

## 4 · LLM Filter Extractor

The LLM receives a query and returns structured JSON filters.
We use an OpenAI-compatible API to call both models.

**System prompt design:**
- Tell the LLM exactly what fields are available (from Section 3)
- Tell it to return ONLY JSON — no extra text
- Tell it to return `null` for fields it is not confident about
- Keep the prompt short — small models work better with focused prompts

**Why extract null when unsure?**
A wrong filter is worse than no filter. If the LLM wrongly adds
`"country": "USA"` to a global query, you will filter out all non-US companies
and miss relevant ones. Conservative extraction is safer.

In [ ]:
# ── Update this based on actual GOI fields found in Section 3 ───────────────
AVAILABLE_FIELDS = {
    'country':           'Country name (e.g. Germany, United States)',
    'state':             'State or region within a country (e.g. Bavaria, California)',
    'municipality':      'City name (e.g. Berlin, Munich, Hamburg)',
    'nace_code':         'NACE industry letter code (e.g. K for software/tech, C for manufacturing, G for retail)',
    'organization_size': 'Company size: Micro (0-9), Small (10-49), Medium (50-249), Large (250+)',
    'keywords':          'Topic keywords to match against pre-extracted summary_keywords field',
}

FIELD_DESCRIPTIONS = '\n'.join(
    f'  - {k}: {v}' for k, v in AVAILABLE_FIELDS.items()
)

SYSTEM_PROMPT = f"""You are a search filter extractor for a company database.
Given a company search query, extract structured filters as a JSON object.

Available filter fields:
{FIELD_DESCRIPTIONS}

NACE code reference (most common):
  A=Agriculture, C=Manufacturing, G=Retail, H=Transport, I=Hospitality,
  J=Media, K=Software/Tech/Telecom, L=Real Estate, M=Consulting/Legal/Science,
  N=Admin Services, O=Government, P=Education, Q=Healthcare

Rules:
1. Return ONLY a valid JSON object. No explanation, no markdown, no backticks.
2. Set a field to null if you are not confident — wrong filters are worse than no filters.
3. keywords must be a list of strings, all other fields are strings or null.
4. Be conservative — only extract what is clearly stated in the query.
5. municipality = city (e.g. Berlin, Munich) — never put a city in country field.
6. country = country only (e.g. Germany, United States) — never put a city here.

Examples:
Query: 'software companies in Germany'
Output: {{"country": "Germany", "state": null, "municipality": null, "nace_code": "K", "organization_size": null, "keywords": ["software"]}}

Query: 'AI startups in Berlin'
Output: {{"country": "Germany", "state": null, "municipality": "Berlin", "nace_code": "K", "organization_size": null, "keywords": ["AI", "artificial intelligence", "machine learning"]}}

Query: 'large pharmaceutical companies'
Output: {{"country": null, "state": null, "municipality": null, "nace_code": null, "organization_size": "Large", "keywords": ["pharmaceutical", "drug", "biotech"]}}

Query: 'manufacturing in Baden-Württemberg'
Output: {{"country": "Germany", "state": "Baden-Württemberg", "municipality": null, "nace_code": "C", "organization_size": null, "keywords": ["manufacturing"]}}

Query: 'renewable energy startups'
Output: {{"country": null, "state": null, "municipality": null, "nace_code": null, "organization_size": "Micro", "keywords": ["renewable energy", "solar", "wind", "clean energy"]}}
"""

def extract_filters(query, model_name, max_retries=3):
    """
    Call local Ollama model to extract structured filters from a query.
    Returns a dict of filters, or empty dict if extraction fails.
    """
    for attempt in range(max_retries):
        try:
            raw = ollama_chat(
                prompt=f'Query: {query}',
                model=model_name,
                system=SYSTEM_PROMPT,
                temperature=0.0,
                max_tokens=150,
            )

            # Clean up common LLM formatting issues
            import re
            raw = re.sub(r'^```json\s*', '', raw)
            raw = re.sub(r'^```\s*',     '', raw)
            raw = re.sub(r'\s*```$',     '', raw)
            raw = raw.strip()

            # Find the JSON object in case LLM adds extra text
            json_match = re.search(r'\{.*\}', raw, re.DOTALL)
            if json_match:
                raw = json_match.group(0)

            filters = json.loads(raw)
            # Remove null/empty fields
            filters = {k: v for k, v in filters.items() if v is not None and v != [] and v != ''}
            return filters

        except (json.JSONDecodeError, Exception) as e:
            if attempt == max_retries - 1:
                print(f'  WARNING: filter extraction failed for "{query}": {e}')
                return {}
            time.sleep(0.5)

# ── Quick test ────────────────────────────────────────────────────────────────
test_queries = [
    'software companies in Germany',
    'AI startups in Berlin',
    'manufacturing in Baden-Württemberg',
    'large pharmaceutical companies',
]
print('=== FILTER EXTRACTION TEST ===')
for q in test_queries:
    lf = extract_filters(q, LLAMA_MODEL)
    mf = extract_filters(q, MINISTRAL_MODEL)
    print(f'Query   : {q}')
    print(f'Llama   : {lf}')
    print(f'Ministral: {mf}')
    print()

## 5 · Extract Filters for All 101 Queries

Run both LLMs over all queries and save results.
We time the extraction so we can include LLM filter time in the total latency.

**Saved to:**
- `result/prefilter/extracted_filters_llama.json`
- `result/prefilter/extracted_filters_ministral.json`

In [ ]:
def extract_all_filters(model_name, label):
    """
    Extract filters for all 101 queries using the given local Ollama model.
    Returns list of dicts with query_id, query, filters, and extraction time.
    """
    results    = []
    total_time = 0

    print(f'Extracting filters with {label} ({model_name})...')
    for i, item in enumerate(data):
        qid   = item['query_id']
        query = item['query']

        t0 = time.perf_counter()
        filters = extract_filters(query, model_name)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        total_time += elapsed_ms

        results.append({
            'query_id':      qid,
            'query':         query,
            'filters':       filters,
            'extraction_ms': round(elapsed_ms, 1),
        })

        if (i + 1) % 10 == 0:
            avg = total_time / (i + 1)
            print(f'  {i+1}/101 done  |  avg {avg:.0f}ms/query')

    avg_ms = total_time / len(results)
    print(f'Done! Average extraction time: {avg_ms:.0f}ms/query')
    return results, avg_ms

# ── Run both models ───────────────────────────────────────────────────────────
llama_filters,     llama_extract_ms     = extract_all_filters(LLAMA_MODEL,     'Llama-3.1-8B')
ministral_filters, ministral_extract_ms = extract_all_filters(MINISTRAL_MODEL, 'Ministral-3B')

# ── Save ─────────────────────────────────────────────────────────────────────
with open(RESULT_DIR / 'extracted_filters_llama.json', 'w') as f:
    json.dump(llama_filters, f, indent=2)

with open(RESULT_DIR / 'extracted_filters_ministral.json', 'w') as f:
    json.dump(ministral_filters, f, indent=2)

print('\nSaved filter files to result/prefilter/')

## 6 · Inspect Extracted Filters

Before applying filters, check what the LLMs actually extracted.
This helps catch problems early — e.g. LLM hallucinating wrong country names,
or being too aggressive/conservative with filters.

In [ ]:
print('=== SAMPLE FILTER COMPARISONS ===')
print(f'{'Query':<45} {'Llama':<40} Ministral')
print('-' * 120)
for i in range(min(15, len(data))):
    q     = llama_filters[i]['query'][:43]
    lf    = str(llama_filters[i]['filters'])[:38]
    mf    = str(ministral_filters[i]['filters'])[:38]
    print(f'{q:<45} {lf:<40} {mf}')

# ── Filter coverage stats ─────────────────────────────────────────────────────
print('\n=== FILTER COVERAGE STATS ===')
for label, filters in [('Llama', llama_filters), ('Ministral', ministral_filters)]:
    field_counts = {}
    empty_count  = 0
    for item in filters:
        if not item['filters']:
            empty_count += 1
        for k in item['filters']:
            field_counts[k] = field_counts.get(k, 0) + 1
    print(f'\n{label}:')
    print(f'  Queries with NO filter extracted: {empty_count}/101')
    for field, count in sorted(field_counts.items(), key=lambda x: -x[1]):
        print(f'  {field:<20}: extracted for {count}/101 queries ({100*count/101:.0f}%)')

## 7 · Apply Filters to Corpus

For each query, use the extracted filters to reduce the corpus to a subset
of matching companies, then run BGE only on that subset.

**Filtering logic:**
- `country`: exact match or case-insensitive contains
- `industry`: case-insensitive contains in industry field or summary
- `naics_code`: exact match on naics_code field (if available)
- `keywords`: at least one keyword must appear in the summary
- If NO filters extracted → fall back to full corpus (same as baseline)
- If filter is too aggressive (< 50 companies remain) → fall back to full corpus

**Why the fallback?** A filter that removes too many companies risks
dropping relevant ones. The fallback ensures recall does not collapse.

In [ ]:
MIN_CORPUS_SIZE = 50   # fall back to full corpus if filter leaves fewer than this

def parse_summary_keywords(val):
    """Parse summary_keywords — stored as string repr of list in CSV."""
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            import ast
            return ast.literal_eval(val)
        except:
            return [val]
    return []

def apply_filters(filters, corpus_df):
    """
    Apply LLM-extracted filters to the corpus using actual GOI structured fields.
    Returns filtered dataframe and original indices.
    """
    if not filters:
        return corpus_df, list(range(len(corpus_df)))

    mask = pd.Series([True] * len(corpus_df), index=corpus_df.index)

    # ── Country filter — exact structured field ───────────────────────────────
    if 'country' in filters and 'country' in corpus_df.columns:
        country = filters['country'].lower()
        mask &= corpus_df['country'].fillna('').str.lower().str.contains(
            country, regex=False
        )

    # ── State / region filter ─────────────────────────────────────────────────
    if 'state' in filters and 'state' in corpus_df.columns:
        state = filters['state'].lower()
        mask &= corpus_df['state'].fillna('').str.lower().str.contains(
            state, regex=False
        )

    # ── City / municipality filter ────────────────────────────────────────────
    if 'municipality' in filters and 'municipality' in corpus_df.columns:
        city = filters['municipality'].lower()
        mask &= corpus_df['municipality'].fillna('').str.lower().str.contains(
            city, regex=False
        )

    # ── NACE code filter — match on just the letter prefix ───────────────────
    # e.g. 'K' matches 'NACE K: Telecommunication, computer programming...'
    if 'nace_code' in filters and 'nace_code' in corpus_df.columns:
        nace_letter = filters['nace_code'].strip().upper()[0]  # just the letter
        mask &= corpus_df['nace_code'].fillna('').str.upper().str.contains(
            f'NACE {nace_letter}', regex=False
        )

    # ── Organization size filter ──────────────────────────────────────────────
    if 'organization_size' in filters and 'organization_size' in corpus_df.columns:
        size = filters['organization_size'].lower()
        mask &= corpus_df['organization_size'].fillna('').str.lower().str.contains(
            size, regex=False
        )

    # ── Keyword filter — match against pre-extracted summary_keywords ─────────
    # Much faster than scanning full summaries!
    if 'keywords' in filters and isinstance(filters['keywords'], list):
        if 'summary_keywords' in corpus_df.columns:
            # Parse the keywords list and check for any match
            parsed_kws = corpus_df['summary_keywords'].apply(parse_summary_keywords)
            kw_mask = parsed_kws.apply(
                lambda company_kws: any(
                    any(fkw.lower() in ckw.lower() for ckw in company_kws)
                    for fkw in filters['keywords']
                )
            )
        else:
            # Fallback to summary scan if summary_keywords not available
            kw_mask = pd.Series([False] * len(corpus_df), index=corpus_df.index)
            for kw in filters['keywords']:
                kw_mask |= corpus_df['summary'].fillna('').str.lower().str.contains(
                    kw.lower(), regex=False
                )
        mask &= kw_mask

    filtered = corpus_df[mask].reset_index(drop=False)
    original_indices = filtered['index'].tolist()
    filtered = filtered.drop(columns=['index'])

    # ── Fallback if too few companies remain ──────────────────────────────────
    if len(filtered) < MIN_CORPUS_SIZE:
        return corpus_df, list(range(len(corpus_df)))

    return filtered, original_indices

# ── Quick test with real GOI fields ──────────────────────────────────────────
print('=== FILTER APPLICATION TESTS ===')
test_cases = [
    {'country': 'Germany', 'nace_code': 'K', 'keywords': ['software']},
    {'country': 'Germany', 'municipality': 'Berlin', 'nace_code': 'K'},
    {'nace_code': 'C', 'state': 'Baden-Württemberg'},
    {'nace_code': 'K', 'organization_size': 'Large'},
]
for tf in test_cases:
    filtered, idx = apply_filters(tf, all_companies)
    fallback = len(filtered) == len(all_companies)
    print(f'Filter: {tf}')
    print(f'  Result: {len(all_companies):,} → {len(filtered):,} companies'
          f'  ({100*len(filtered)/len(all_companies):.1f}%)'
          f'  {"[FALLBACK]" if fallback else ""}')
    print()

## 8 · Full Retrieval Pipeline with Pre-filter

For each query:
1. Apply LLM-extracted filters → get filtered corpus
2. Retrieve pre-computed BGE embeddings for the filtered companies
3. Build a small FAISS index on the fly for the filtered subset
4. Encode the query with BGE and search the small index
5. Return top-1000 results (or all results if filtered corpus < 1000)

**Why build the FAISS index on the fly?**
Because each query filters to a *different* subset of companies,
we cannot pre-build one index. But since the filtered corpus is small
(e.g. 3,000 instead of 99,000), building a flat index takes milliseconds.

**Timing:** We record total query time = filter apply + index build + encode + search.

In [ ]:
def run_prefilter_pipeline(filter_results, label, model_name):
    """
    Run the full pre-filter + BGE retrieval pipeline for all 101 queries.
    Returns results dataframe and timing stats.
    """
    all_rows       = []
    timing_rows    = []
    corpus_sizes   = []

    print(f'Running pre-filter pipeline with {label}...')

    for item_filters in filter_results:
        qid     = item_filters['query_id']
        query   = item_filters['query']
        filters = item_filters['filters']
        extract_ms = item_filters['extraction_ms']

        # ── Step 1: Apply filters ──────────────────────────────────────────────
        t0 = time.perf_counter()
        filtered_df, orig_indices = apply_filters(filters, all_companies)
        filter_apply_ms = (time.perf_counter() - t0) * 1000
        corpus_sizes.append(len(filtered_df))

        # ── Step 2: Get embeddings for filtered subset ─────────────────────────
        t0 = time.perf_counter()
        subset_embs = all_embeddings[orig_indices].astype('float32')
        emb_slice_ms = (time.perf_counter() - t0) * 1000

        # ── Step 3: Build small FAISS index ───────────────────────────────────
        t0 = time.perf_counter()
        dim          = subset_embs.shape[1]  # 1024
        small_index  = faiss.IndexFlatIP(dim)
        small_index.add(subset_embs)
        index_build_ms = (time.perf_counter() - t0) * 1000

        # ── Step 4: Encode query ──────────────────────────────────────────────
        t0 = time.perf_counter()
        q_emb = model_bge.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype('float32')
        encode_ms = (time.perf_counter() - t0) * 1000

        # ── Step 5: Search ────────────────────────────────────────────────────
        t0 = time.perf_counter()
        k_retrieve   = min(1000, len(filtered_df))
        scores, idxs = small_index.search(q_emb, k_retrieve)
        search_ms    = (time.perf_counter() - t0) * 1000

        # ── Collect results ───────────────────────────────────────────────────
        for rank, (local_idx, score) in enumerate(zip(idxs[0], scores[0])):
            company = filtered_df.iloc[local_idx]
            all_rows.append({
                'query_id': qid,
                'query':    query,
                'rank':     rank + 1,
                'score':    float(score),
                'domain':   company['domain'],
                'name':     company.get('name', ''),
                'summary':  company['summary'],
                'filters_applied': json.dumps(filters),
                'corpus_size_after_filter': len(filtered_df),
            })

        timing_rows.append({
            'query_id':          qid,
            'query':             query,
            'llm_extract_ms':    extract_ms,
            'filter_apply_ms':   round(filter_apply_ms, 1),
            'emb_slice_ms':      round(emb_slice_ms, 1),
            'index_build_ms':    round(index_build_ms, 1),
            'encode_ms':         round(encode_ms, 1),
            'search_ms':         round(search_ms, 1),
            'total_online_ms':   round(filter_apply_ms + index_build_ms + encode_ms + search_ms, 1),
            'corpus_size':       len(filtered_df),
            'fallback_used':     len(filtered_df) == len(all_companies),
        })

    results_df = pd.DataFrame(all_rows)
    timing_df  = pd.DataFrame(timing_rows)

    # ── Save ──────────────────────────────────────────────────────────────────
    results_df.to_csv(RESULT_DIR / f'retrieval_{label.lower()}.csv', index=False)
    timing_df.to_csv(RESULT_DIR  / f'timing_{label.lower()}.csv', index=False)

    avg_corpus = np.mean(corpus_sizes)
    avg_total  = timing_df['total_online_ms'].mean()
    fallbacks  = timing_df['fallback_used'].sum()

    print(f'  Done!')
    print(f'  Avg corpus size after filter : {avg_corpus:,.0f} / {len(all_companies):,} companies ({100*avg_corpus/len(all_companies):.1f}%)')
    print(f'  Avg online query latency     : {avg_total:.0f}ms (excludes LLM extraction)')
    print(f'  Fallback to full corpus      : {fallbacks}/101 queries')
    print(f'  Saved to result/prefilter/retrieval_{label.lower()}.csv')

    return results_df, timing_df

# ── Run both pipelines ────────────────────────────────────────────────────────
llama_results_df,     llama_timing_df     = run_prefilter_pipeline(llama_filters,     'llama',     'llama-3.1-8b')
ministral_results_df, ministral_timing_df = run_prefilter_pipeline(ministral_filters, 'ministral', 'ministral-3b')

## 9 · Evaluation — Precision, Recall, NDCG@k

Evaluate both pre-filter pipelines using the same metrics and pseudo-relevance
labels as the baseline experiments, so results are directly comparable.

In [ ]:
K_VALUES = [10, 50, 100, 1000]

def get_relevant(query_id, top_k=100):
    return set(production_df[
        (production_df['query_id'] == query_id) &
        (production_df['rank'] <= top_k)
    ]['domain'].tolist())

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k if k else 0

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0

def dcg_at_k(retrieved, relevant, k):
    return sum(1 / np.log2(i + 2) for i, d in enumerate(retrieved[:k]) if d in relevant)

def ndcg_at_k(retrieved, relevant, k):
    ideal = dcg_at_k(list(relevant), relevant, k)
    return dcg_at_k(retrieved, relevant, k) / ideal if ideal else 0

def evaluate_results(results_df, label):
    rows = []
    for item in data:
        qid      = item['query_id']
        query    = item['query']
        relevant = get_relevant(qid)
        retrieved = (
            results_df[results_df['query_id'] == qid]
            .sort_values('rank')['domain']
            .tolist()
        )
        for k in K_VALUES:
            rows.append({
                'method':    label,
                'query_id':  qid,
                'query':     query,
                'k':         k,
                'precision': precision_at_k(retrieved, relevant, k),
                'recall':    recall_at_k(retrieved, relevant, k),
                'ndcg':      ndcg_at_k(retrieved, relevant, k),
            })
    df = pd.DataFrame(rows)
    df.to_csv(RESULT_DIR / f'evaluation_{label.lower()}.csv', index=False)
    return df

llama_eval_df     = evaluate_results(llama_results_df,     'Llama')
ministral_eval_df = evaluate_results(ministral_results_df, 'Ministral')
print('Evaluation complete!')

## 10 · Results Comparison

Compare both pre-filter pipelines against the BGE baseline (no filter).
This is the key table for the thesis — does pre-filtering actually help?

In [ ]:
# ── Load BGE baseline for comparison ─────────────────────────────────────────
bge_baseline_df = pd.read_csv('result/evaluation_fixed.csv')
bge_baseline_df = bge_baseline_df[bge_baseline_df['method'] == 'BGE']

# ── Build comparison table ────────────────────────────────────────────────────
all_evals = {
    'BGE (no filter)':      bge_baseline_df,
    'BGE + Llama filter':   llama_eval_df,
    'BGE + Ministral filter': ministral_eval_df,
}

comparison_rows = []
print('=== COMPARISON: PRE-FILTER vs BASELINE ===')
print(f'{'Method':<28} {'k':>6} | {'NDCG':>7} | {'Prec':>7} | {'Recall':>7}')
print('=' * 65)

for method_name, eval_df in all_evals.items():
    for k in K_VALUES:
        subset = eval_df[eval_df['k'] == k]
        ndcg   = subset['ndcg'].mean()
        prec   = subset['precision'].mean()
        rec    = subset['recall'].mean()
        print(f'{method_name:<28} {k:>6} | {ndcg:>7.3f} | {prec:>7.3f} | {rec:>7.3f}')
        comparison_rows.append({
            'method': method_name, 'k': k,
            'ndcg': round(ndcg, 3),
            'precision': round(prec, 3),
            'recall': round(rec, 3),
        })
    print('-' * 65)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(RESULT_DIR / 'comparison_prefilter.csv', index=False)
print('\nSaved to result/prefilter/comparison_prefilter.csv')

## 11 · Latency Comparison

Compare total query latency with and without pre-filtering.

**Important:** Pre-filtering adds two new latency components:
- **LLM extraction time** — calling the LLM to extract filters
- **Filter apply time** — applying the filter rules to the corpus
- **Index build time** — building a small FAISS index on the fly

But it reduces:
- **Search time** — searching a smaller corpus is faster

Net effect depends on how much the corpus is reduced.

In [ ]:
BGE_BASELINE_LATENCY_MS = 44.4  # from baseline experiment

print('=== LATENCY BREAKDOWN ===')
print(f'{'Method':<28} {'LLM (ms)':>10} {'Filter (ms)':>12} {'Index (ms)':>11} {'Encode (ms)':>12} {'Search (ms)':>12} {'Total* (ms)':>12}')
print('=' * 100)

# BGE baseline (no filter)
print(f'{'BGE (no filter)':<28} {'N/A':>10} {'N/A':>12} {'N/A':>11} {'16.6':>12} {'27.8':>12} {BGE_BASELINE_LATENCY_MS:>12.1f}')

for label, timing_df in [('BGE + Llama filter', llama_timing_df),
                           ('BGE + Ministral filter', ministral_timing_df)]:
    llm_ms    = timing_df['llm_extract_ms'].mean()
    filter_ms = timing_df['filter_apply_ms'].mean()
    index_ms  = timing_df['index_build_ms'].mean()
    encode_ms = timing_df['encode_ms'].mean()
    search_ms = timing_df['search_ms'].mean()
    total_ms  = llm_ms + filter_ms + index_ms + encode_ms + search_ms
    print(f'{label:<28} {llm_ms:>10.1f} {filter_ms:>12.1f} {index_ms:>11.1f} {encode_ms:>12.1f} {search_ms:>12.1f} {total_ms:>12.1f}')

print('\n* Total includes LLM extraction time')
print('  For production, LLM could run in parallel with other pipeline steps')

# ── Corpus size reduction ─────────────────────────────────────────────────────
print('\n=== CORPUS SIZE REDUCTION ===')
for label, timing_df in [('Llama', llama_timing_df), ('Ministral', ministral_timing_df)]:
    avg_size     = timing_df['corpus_size'].mean()
    min_size     = timing_df['corpus_size'].min()
    max_size     = timing_df['corpus_size'].max()
    fallback_n   = timing_df['fallback_used'].sum()
    reduction    = 100 * (1 - avg_size / len(all_companies))
    print(f'{label}: avg={avg_size:,.0f}  min={min_size:,}  max={max_size:,}  '
          f'reduction={reduction:.0f}%  fallbacks={fallback_n}/101')

## 12 · Visualisations

Three plots:
1. NDCG@k comparison across all three pipelines
2. Precision@k comparison
3. Corpus size distribution after filtering (per query)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors  = {'BGE (no filter)': '#4CAF50',
            'BGE + Llama filter': '#2196F3',
            'BGE + Ministral filter': '#FF9800'}
markers = {'BGE (no filter)': '^',
            'BGE + Llama filter': 'o',
            'BGE + Ministral filter': 's'}

plot_data = {
    'BGE (no filter)':        bge_baseline_df,
    'BGE + Llama filter':     llama_eval_df,
    'BGE + Ministral filter': ministral_eval_df,
}

# ── NDCG ─────────────────────────────────────────────────────────────────────
ax = axes[0]
for name, df in plot_data.items():
    vals = [df[df['k']==k]['ndcg'].mean() for k in K_VALUES]
    ax.plot(K_VALUES, vals, marker=markers[name], color=colors[name],
            label=name, linewidth=2, markersize=7)
ax.set_xscale('log'); ax.set_xticks(K_VALUES); ax.set_xticklabels(K_VALUES)
ax.set_xlabel('k'); ax.set_ylabel('NDCG@k')
ax.set_title('NDCG@k: Pre-filter vs Baseline', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── PRECISION ────────────────────────────────────────────────────────────────
ax = axes[1]
for name, df in plot_data.items():
    vals = [df[df['k']==k]['precision'].mean() for k in K_VALUES]
    ax.plot(K_VALUES, vals, marker=markers[name], color=colors[name],
            label=name, linewidth=2, markersize=7)
ax.set_xscale('log'); ax.set_xticks(K_VALUES); ax.set_xticklabels(K_VALUES)
ax.set_xlabel('k'); ax.set_ylabel('Precision@k')
ax.set_title('Precision@k: Pre-filter vs Baseline', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── CORPUS SIZE DISTRIBUTION ─────────────────────────────────────────────────
ax = axes[2]
ax.hist(llama_timing_df['corpus_size'],     bins=20,
        alpha=0.6, color='#2196F3', label='Llama filter')
ax.hist(ministral_timing_df['corpus_size'], bins=20,
        alpha=0.6, color='#FF9800', label='Ministral filter')
ax.axvline(len(all_companies), color='#4CAF50', linestyle='--',
           linewidth=2, label=f'No filter ({len(all_companies):,})')
ax.set_xlabel('Companies after filter')
ax.set_ylabel('Number of queries')
ax.set_title('Corpus Size After Pre-filter\n(per query)', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULT_DIR / 'prefilter_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved to result/prefilter/prefilter_comparison.png')

## 13 · Which LLM Won? Final Decision

Based on quality metrics, latency, and filter quality — which LLM should
we use for all future experiments?

In [ ]:
print('=== FINAL DECISION ===')
print()

metrics = ['ndcg', 'precision', 'recall']
scores  = {'Llama': {}, 'Ministral': {}}

for k in K_VALUES:
    l_sub = llama_eval_df[llama_eval_df['k'] == k]
    m_sub = ministral_eval_df[ministral_eval_df['k'] == k]
    for metric in metrics:
        scores['Llama'][f'{metric}@{k}']     = l_sub[metric].mean()
        scores['Ministral'][f'{metric}@{k}'] = m_sub[metric].mean()

# Count wins
llama_wins     = 0
ministral_wins = 0
for key in scores['Llama']:
    l = scores['Llama'][key]
    m = scores['Ministral'][key]
    if l > m:
        llama_wins += 1
        print(f'  {key:<20}: Llama={l:.3f}  Ministral={m:.3f}  ← Llama wins')
    elif m > l:
        ministral_wins += 1
        print(f'  {key:<20}: Llama={l:.3f}  Ministral={m:.3f}  ← Ministral wins')
    else:
        print(f'  {key:<20}: Llama={l:.3f}  Ministral={m:.3f}  ← Tied')

print(f'\nQuality wins  → Llama: {llama_wins}  Ministral: {ministral_wins}')

llama_lat     = (llama_timing_df['llm_extract_ms'] + llama_timing_df['total_online_ms']).mean()
ministral_lat = (ministral_timing_df['llm_extract_ms'] + ministral_timing_df['total_online_ms']).mean()
print(f'Avg total latency → Llama: {llama_lat:.0f}ms  Ministral: {ministral_lat:.0f}ms')

if llama_wins >= ministral_wins:
    print('\n✅ WINNER: Llama-3.1-8B — use for all future experiments')
    winner = 'llama'
else:
    print('\n✅ WINNER: Ministral-3B — use for all future experiments')
    winner = 'ministral'

# Save the winner label for future notebooks
with open(RESULT_DIR / 'winner_llm.txt', 'w') as f:
    f.write(winner)
print(f'Winner saved to result/prefilter/winner_llm.txt')